## Домашнее задание по теме «Архитектуры свёрточных сетей»

Подберите базовую модель для работы по вашей задаче. Попробуйте обучить различные модели на «ваших» данных. По результатам выберите лучшую модель для дальнейшего обучения.

Задание
Проведите эксперименты по начальному обучению различных моделей и сравните результаты.

Возьмите датасет EMNIST из torchvision.
Обучите на нём модели: ResNet 18, VGG 16, Inception v3, DenseNet 161:
желательно обучить каждую модель с нуля по 10 эпох
если ресурсов компьютера / Colab не хватает, достаточно обучить каждую модель по 1-2 эпохи
Сведите результаты обучения моделей (графики лосса) в таблицу и сравните их.

In [ ]:
import torch
import torchvision
from torchvision.transforms import ToTensor, Compose, Resize
import matplotlib.pyplot as plt
from torchsummary import summary
import time

# EMNIST по умолчанию 28x28. ResNet18 ожидает 224x224, поэтому меняем размер
transform = Compose([
    Resize((224, 224)),  # увеличиваем картинку до 224x224
    ToTensor(),
    # Нормализация для ResNet (ImageNet stats)
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                     std=[0.229, 0.224, 0.225])
])

# Загрузка тренировочного датасета
train_dataset = torchvision.datasets.EMNIST(
    root='./3.data',  # Корневой каталог для хранения данных
    split='balanced', # Сбалансированное подмножество, где количество примеров примерно одинаково для каждого класса
    train=True,    # True для обучающей выборки, False для тестовой
    download=True, # Если True, данные будут скачаны из интернета
    transform=transform  # Преобразование данных в тензоры + подшаманили датасет
)

# Загрузка тестового датасета
test_dataset = torchvision.datasets.EMNIST(
    root='./2.data',
    split='balanced',
    train=False,    # False для тестовой
    download=True,
    transform=transform
)

print(f'Кол-во изображений для тренировки {len(train_dataset)}, для теста {len(test_dataset)}')
print(f'Кол-во классов тренировки {len(train_dataset.classes)}, для теста {len(test_dataset.classes)}')



Кол-во изображений для тренировки 112800, для теста 18800
Кол-во классов тренировки 47, для теста 47


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time

# ============================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================

from torchvision.transforms import Compose, Resize, ToTensor, Normalize

def get_dataloaders(batch_size=256, img_size=224):
    # ПРАВИЛЬНАЯ нормализация для предобученных моделей (ImageNet stats)
    # Для grayscale мы используем среднее по всем трем каналам
    transform = Compose([
        Resize((img_size, img_size)),
        ToTensor(),
        # усредненные значения по RGB
        # Normalize(mean=[0.485, 0.456, 0.406], 
        #           std=[0.229, 0.224, 0.225])
        # ДЛЯ GRAYSCALE: одно значение для одного канала
        Normalize(mean=[0.485], std=[0.229])
    ])
    
    train_dataset = torchvision.datasets.EMNIST(
        root='./3.data',
        split='balanced',
        train=True,
        download=True,
        transform=transform
    )
    
    test_dataset = torchvision.datasets.EMNIST(
        root='./3.data',
        split='balanced',
        train=False,
        download=True,
        transform=transform
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader

# ============================
# 2. СОЗДАНИЕ МОДЕЛЕЙ (АДАПТАЦИЯ)
# ============================
def create_resnet18(num_classes=47, in_channels=1):
    # Адаптируем первый слой (вместо 3 каналов — 1 канал)
    # И меняем ядро с 7x7 на 3x3, потому что картинка маленькая (28x28)
    model = torchvision.models.resnet18(pretrained=True)
    model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Убираем агрессивный пулинг, чтобы не потерять информацию
    model.maxpool = nn.Identity()
    # Меняем последний слой (вместо 1000 классов — 47)
    model.fc = nn.Linear(512, num_classes)
    return model

def create_vgg16(num_classes=47, in_channels=1):
    model = torchvision.models.vgg16(pretrained=True)
    model.features[0] = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
    num_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(num_features, num_classes)
    return model

def create_densenet161(num_classes=47, in_channels=1):
    model = torchvision.models.densenet161(pretrained=True)
    model.features.conv0 = nn.Conv2d(in_channels, 96, kernel_size=3, stride=1, padding=1, bias=False)
    model.features.pool0 = nn.Identity()
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, num_classes)
    return model

def create_inception_v3(num_classes=47, in_channels=1):
    model = torchvision.models.inception_v3(pretrained=True, aux_logits=False)
    model.Conv2d_1a_3x3 = nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool1 = nn.Identity()
    model.maxpool2 = nn.Identity()
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    return model

# ============================
# 3. УНИВЕРСАЛЬНАЯ ФУНКЦИЯ ОБУЧЕНИЯ
# ============================
def train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs, device, model_name="Model"):
    model = model.to(device)
    
    train_losses = []
    test_losses = []
    test_accuracies = []
    
    print(f"Начинаем обучение: {model_name}")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Фаза обучения
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        avg_train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        
        # Фаза тестирования
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                test_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()
        
        avg_test_loss = test_loss / test_total
        test_acc = test_correct / test_total
        
        train_losses.append(avg_train_loss)
        test_losses.append(avg_test_loss)
        test_accuracies.append(test_acc)
        
        print(f"Эпоха {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f} | "
              f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    elapsed_time = time.time() - start_time
    print(f"{model_name} завершена за {elapsed_time:.2f} сек.")
    
    return {
        'name': model_name,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'time': elapsed_time
    }

# ============================
# 4. ОБУЧЕНИЕ ВСЕХ МОДЕЛЕЙ
# ============================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Устройство: {device}")

num_epochs = 10
batch_size = 64 # экономия памяти
criterion = nn.CrossEntropyLoss()

train_loader, test_loader = get_dataloaders(batch_size=batch_size, img_size=128)  # увеличиваем картинку до 128 (модели ожидают 224x224 но для экономии памяти не будем)

models_config = {
    'ResNet18': create_resnet18,
    'VGG16': create_vgg16,
    'DenseNet161': create_densenet161,
    'Inception_v3': create_inception_v3
}

results = {}

for name, creator in models_config.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name}")
    print('='*50)
    
    model = creator()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    results[name] = train_model(
        model, train_loader, test_loader, optimizer, criterion,
        num_epochs, device, model_name=name
    )

    # === ОСВОБОЖДАЕМ ПАМЯТЬ ===
    print(f"Очистка памяти после {name}...")
    del model
    del optimizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================
# 5. СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
# ============================
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)

print(f"{'Модель':<15} | {'Test Loss':<12} | {'Test Acc':<10} | {'Время (сек)':<12}")
print("-"*60)

for name, data in results.items():
    print(f"{name:<15} | {data['test_losses'][-1]:<12.4f} | {data['test_accuracies'][-1]:<10.4f} | {data['time']:<12.2f}")

# ============================
# 6. ВИЗУАЛИЗАЦИЯ ГРАФИКОВ
# ============================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График потерь (Loss)
for name, data in results.items():
    axes[0].plot(data['train_losses'], label=f'{name} (train)', linestyle='--')
    axes[0].plot(data['test_losses'], label=f'{name} (test)')
axes[0].set_title('Функция потерь (Loss)')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# График точности (Accuracy)
for name, data in results.items():
    axes[1].plot(data['test_accuracies'], label=name, marker='o')
axes[1].set_title('Точность на тестовой выборке (Accuracy)')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()

print("Графики сохранены в 'model_comparison.png'")

Устройство: cpu

Модель: ResNet18
Начинаем обучение: ResNet18
